# Configuração do Ambiente Virtual

Siga as instruções abaixo para configurar o ambiente deste notebook:

1. **Execute a célula de código abaixo.** Ela usará o `uv` para criar um ambiente virtual (`.venv`) e instalará as bibliotecas necessárias, incluindo o `ipykernel`.
2. **Execute um Reload Window** (Ctrl+Shift+P → Developer: Reload Window).
3. **Troque o Kernel do Notebook no VS Code:**
   * Após a execução concluir, clique no nome do Kernel atual, no **canto superior direito** da janela do notebook.
   * Escolha **"Select Another Kernel..."** (Selecionar Outro Kernel).
   * Selecione **"Python Environments"** (Ambientes Python).
   * Escolha o Python que está dentro da pasta `.venv` que acabou de ser criada (ex: `./.venv/bin/python`).
4. **Pronto!** Agora você pode executar as próximas células de análise de dados sem problemas de dependências.

In [ ]:
# Instala o uv (caso não esteja instalado no ambiente base)
%pip install uv

# Cria o ambiente virtual (.venv) no diretório atual e força a substituição se já existir
!uv venv --clear # Voce pode desabilitar essa linha se ja tiver o .venv no projeto.

# Instala as bibliotecas necessárias
!uv pip install pandas numpy matplotlib scikit-learn imbalanced-learn


In [ ]:
!uv pip install ipykernel

In [ ]:
!uv run python -m ipykernel install --user --name aula-04 --display-name "cursos-eng-dados-ibm-churn(uv)"


# Data Preparation Pipeline for Machine Learning
## Case Study: IBM Telco Customer Churn (Kaggle)

> **Contexto:** Dataset real disponibilizado pela IBM. Contém clientes de uma empresa de Telecomunicações que assinam serviços de Internet e Telefone. Nosso objetivo é prever quem **cancelou a assinatura** (Target = `Churn`: `Yes`/`No`).

Etapas:
1. Análise dos Dados (EDA)
2. Limpeza de Dados (coerção de `TotalCharges` + remoção de irrelevantes + imputação)
3. Tratamento de Outliers
4. Transformações Estatísticas (StandardScaler)
5. Encoding de Variáveis Categóricas (mapeamento binário + One-Hot Encoding)
6. Seleção de Features
7. Split do Dataset e Balanceamento com SMOTE


![imagem](../aula-01-titanic/image/pipeline-pre-processamento.jpg)


A imagem divide o pipeline em duas macrofases (Fase 1: Exploração e Saneamento / Fase 2: Engenharia e Estruturação) e abriga os 7 passos descritos da seguinte forma:

Na **Fase 1: Exploração e Saneamento**:

*   A bolha amarela do topo (**EDA e Limpeza Inicial**): Contempla perfeitamente as etapas **1 (Análise dos Dados - EDA)** e **2 (Limpeza de Dados)**.
*   A bolha verde-água inferior (**Tratamento de Outliers**): Reflete de forma direta a etapa **3 (Tratamento de Outliers)**.

Na **Fase 2: Engenharia e Estruturação**:

*   A engrenagem laranja superior (**Transformações e Encoding**): Agrupa as etapas **4 (Transformações Estatísticas)** e **5 (Encoding de Variáveis Categóricas)**.
*   A balança vermelha inferior (**Seleção e Split Final**): Sintetiza a etapa **6 (Seleção de Features)** e a etapa **7 (Split do Dataset e Balanceamento)**.

Por fim, a etapa resultante pós-split é direcionada para **"Treinamento"** (com balanceamento via SMOTE) e **"Teste"**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif

from imblearn.over_sampling import SMOTE

# Carregar Dataset

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

# 1. Análise Exploratória (EDA)

**Observação do Professor:**
Para iniciarmos nossa EDA, o primeiro passo é entender a estrutura dos dados.

O dataset IBM Telco é chamado de **"O Rei do Encoding"** porque transborda variáveis categóricas. A função `df.info()` revelará o dtype de cada coluna. Preste atenção especial em:

- **`customerID`**: Identificador único — sem valor preditivo. Será removido.
- **`TotalCharges`**: ⚠️ **Armadilha real!** Apesar de ser um valor financeiro numérico, o Pandas a carregará como `object` (string/texto). Isso ocorre porque clientes recém-cadastrados que ainda não geraram fatura têm o campo preenchido com um **espaço em branco** (`" "`). O Pandas, ao encontrar texto misturado com números, converte a coluna inteira para string. Resolveremos isso na limpeza com `pd.to_numeric(errors='coerce')`.
- **Colunas dicotômicas** (`OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`): Possuem 3 categorias: `Yes`, `No` e `No internet service`. Um detalhe sutil que exige atenção no encoding.
- **`Churn`**: Nossa **variável alvo** (`Yes`/`No`). Dataset **levemente desbalanceado**: ~73% `No` e ~27% `Yes`.

In [ ]:
df.info()

**Dicionário de Dados (Significado das Features):**

*   **`0. customerID`**: ID único do cliente. Sem valor preditivo — será removido.
*   **`1. gender`**: Gênero do cliente (`Male`/`Female`). Variável **categórica binária**.
*   **`2. SeniorCitizen`**: Se o cliente é idoso (`1` = sim / `0` = não). Variável **binária** — já está em formato numérico.
*   **`3. Partner`**: Se o cliente tem cônjuge (`Yes`/`No`). Variável **categórica binária**.
*   **`4. Dependents`**: Se o cliente tem dependentes (`Yes`/`No`). Variável **categórica binária**.
*   **`5. tenure`**: Quantos meses o cliente está com a empresa. Variável **numérica** — alta correlação com churn (clientes mais novos cancelam mais).
*   **`6. PhoneService`**: Se tem serviço de telefone (`Yes`/`No`). Variável **categórica binária**.
*   **`7. MultipleLines`**: Se tem múltiplas linhas (`Yes`/`No`/`No phone service`). Variável **categórica** com 3 categorias.
*   **`8. InternetService`**: Tipo de serviço de internet (`DSL`/`Fiber optic`/`No`). Variável **categórica nominal** — exige OHE.
*   **`9. OnlineSecurity`**: Se tem segurança online (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`10. OnlineBackup`**: Se tem backup online (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`11. DeviceProtection`**: Se tem proteção de dispositivo (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`12. TechSupport`**: Se tem suporte técnico (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`13. StreamingTV`**: Se assina streaming de TV (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`14. StreamingMovies`**: Se assina streaming de filmes (`Yes`/`No`/`No internet service`). Variável **categórica** com 3 categorias.
*   **`15. Contract`**: Tipo de contrato (`Month-to-month`/`One year`/`Two year`). Variável **categórica ordinal** — exige OHE.
*   **`16. PaperlessBilling`**: Se usa faturamento sem papel (`Yes`/`No`). Variável **categórica binária**.
*   **`17. PaymentMethod`**: Método de pagamento (4 categorias). Variável **categórica nominal** — exige OHE.
*   **`18. MonthlyCharges`**: Valor da mensalidade. Variável **numérica** — requer StandardScaler.
*   **`19. TotalCharges`**: ⚠️ Total cobrado até hoje. Carregado como **string** pelo Pandas — requer coerção numérica + tratamento de nulos.
*   **`20. Churn`**: Nossa **variável alvo** (`Yes` = cancelou / `No` = permaneceu).

**Observação do Professor:**
Antes de rodar `df.describe()`, precisamos converter `TotalCharges` para numérico (faremos isso na limpeza). Por ora, vamos inspecionar a distribuição do target e identificar o desbalanceamento.

In [ ]:
print('Distribuição do Target (Churn):')
print(df['Churn'].value_counts())
print(f'\nPercentual de Churn: {(df["Churn"] == "Yes").mean()*100:.1f}%')

**Observação do Professor:**
A função `df.isnull().sum()` deve mostrar **zero nulos** neste ponto — mas isso é enganoso! Os nulos de `TotalCharges` estão disfarçados como espaços em branco (` `), razão pela qual o Pandas os trata como strings válidas. A verdadeira limpeza ocorre na próxima etapa.

In [ ]:
print('Valores nulos (antes da coerção de TotalCharges):')
print(df.isnull().sum())
print()
print(f'Dtype de TotalCharges: {df["TotalCharges"].dtype}')
print(f'Exemplos de valores problemáticos: {df[df["TotalCharges"].str.strip() == ""]["TotalCharges"].head().tolist()}')

# Visualização

**Observação do Professor:**
Plotamos os histogramas das features numéricas já disponíveis. Note:
- **`tenure`**: Distribuição bimodal — concentração de clientes muito novos (possíveis churners) e muito antigos (clientes fiéis).
- **`MonthlyCharges`**: Multimodal — reflete os diferentes planos (internet DSL, fibra óptica, pacotes combinados).
- **`SeniorCitizen`**: Binária (0/1) — apenas ~16% são idosos, mas este grupo tem taxa de churn maior.

In [ ]:
df[['tenure', 'MonthlyCharges', 'SeniorCitizen']].hist(figsize=(12, 4))
plt.tight_layout()
plt.show()

# 2. Limpeza de Dados

**Observação do Professor:**
Esta é a etapa mais rica e desafiadora deste dataset. Temos 3 problemas para resolver:

**Problema 1 — Coluna irrelevante:** `customerID` é um identificador único sem poder preditivo.

**Problema 2 — `TotalCharges` como string (o problema real do cotidiano):**
O campo `TotalCharges` foi exportado do sistema IBM com **espaços em branco** para clientes sem histórico de cobrança. Usamos `pd.to_numeric(errors='coerce')` para forçar a conversão:
- Valores numéricos válidos → convertidos normalmente.
- Espaços em branco e strings inválidas → convertidos para `NaN` (nulo).

Após a conversão, os NaNs revelados são tratados com a **mediana** (robusta a outliers).

**Problema 3 — Target como string:** `Churn` está como `Yes`/`No`. Convertemos para `1`/`0` para os algoritmos de ML.

**Problema 4 — Remoção de duplicatas:** Boa prática padrão.

In [ ]:
# Remove identificador sem valor preditivo
df = df.drop(columns=['customerID'])

# ⚠️ Coerção de TotalCharges: força conversão para numérico, espaços viram NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Imputa os NaNs de TotalCharges com a mediana
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Converte o target para 0/1
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

# Remove duplicatas
df = df.drop_duplicates()

print(f'Shape após limpeza: {df.shape}')
print(f'Nulos restantes: {df.isnull().sum().sum()}')
print(f'Dtype de TotalCharges: {df["TotalCharges"].dtype}')

**Observação do Professor — Por que `pd.to_numeric(errors='coerce')` é uma habilidade essencial?**

Este cenário — uma coluna numérica carregada como string por causa de valores inválidos — é **extremamente comum na vida real**. Sistemas legados, exportações de ERPs, relatórios de BI: todos podem gerar esse tipo de problema.

O parâmetro `errors='coerce'` é a chave:
- `errors='raise'` (padrão): lança um erro ao encontrar um valor inválido → para a execução.
- `errors='ignore'`: mantém o valor original como string → não resolve o problema.
- **`errors='coerce'`**: converte inválidos para `NaN` → permite tratamento controlado.

Após a coerção, os NaNs ficam visíveis e podemos aplicar nossa estratégia de imputação. **Esse padrão de detecção → coerção → imputação é um dos mais importantes do arsenal de um Engenheiro de Dados.**

💡 **Regra de Ouro:** Sempre verifique o dtype das colunas numéricas com `df.info()`. Se uma coluna que deveria ser `float64` aparecer como `object`, suspeite de valores problemáticos escondidos.

# 3. Tratamento de Outliers

**Observação do Professor - Tratamento de Outliers (Método IQR):**

As features numéricas deste dataset (`tenure`, `MonthlyCharges`, `TotalCharges`) são relativamente bem-comportadas — não apresentam a assimetria extrema do Spaceship Titanic ou as escalas absurdas do Churn Bancário. No entanto, sempre verificamos outliers como boa prática.

Aplicamos o **Método IQR** em `MonthlyCharges` e `TotalCharges`:
- **`tenure`**: varia de forma natural de 0 a 72 meses — sem outliers reais.
- **`MonthlyCharges`**: alguns planos premium podem gerar valores altos mas legítimos; o IQR ajuda a identificar anomalias genuínas.
- **`TotalCharges`**: é o produto de `tenure × MonthlyCharges` — outliers aqui geralmente refletem combinações incomuns de plano + tempo de contrato.

![Tratamento de Outliers (IQR)](../aula-01-prepa-dados/image/outliers.jpg)

In [ ]:
outlier_cols = ['MonthlyCharges', 'TotalCharges']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)  # calcula o primeiro quartil
    Q3 = df[col].quantile(0.75)  # calcula o terceiro quartil
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR      # margem de tolerância inferior
    upper = Q3 + 1.5 * IQR      # margem de tolerância superior
    df = df[(df[col] >= lower) & (df[col] <= upper)]

print(f'Shape após remoção de outliers: {df.shape}')

# 4. Transformações Estatísticas

**Observação do Professor:**
Neste dataset **não aplicamos transformação logarítmica** — as distribuições não apresentam a assimetria extrema que justificaria essa técnica.

Aplicamos o `StandardScaler` nas features numéricas contínuas para padronizar as escalas. As features binárias (`SeniorCitizen`, `HasCrCard`-equivalentes) não precisam de padronização pois já estão na escala [0, 1].

Após o `StandardScaler`, cada feature numérica terá média ≈ 0 e desvio padrão ≈ 1, colocando `tenure` (0–72), `MonthlyCharges` (~18–118) e `TotalCharges` (~18–8684) na mesma escala relativa.

In [ ]:
numeric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

df[numeric_features].describe().round(2)

# 5. Encoding

**Observação do Professor — O Rei do Encoding:**
Este dataset é chamado de **"O Rei do Encoding"** por uma boa razão: ele possui uma densidade raramente vista de variáveis categóricas que precisam ser tratadas de formas distintas.

**Estratégia adotada:**

**1. Mapeamento Binário** (`Yes`/`No` → `1`/`0`):
Aplicado em colunas com exatamente 2 valores (`Yes`/`No`). Não precisamos de OHE aqui — um único número já representa ambos os estados sem ambiguidade:
`gender`, `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`

**2. Atenção especial às colunas com 3 categorias:**
Colunas como `OnlineSecurity`, `OnlineBackup`, `TechSupport`, etc. têm um terceiro valor: `No internet service`. Isso significa que o cliente **não tem internet**, portanto logicamente também não tem o serviço em questão. Mapeamos `No internet service` → `0` (equivalente a `No`) antes do OHE.

**3. One-Hot Encoding** (OHE com `drop_first=True`):
Aplicado em variáveis nominais com múltiplas categorias sem ordem natural:
`MultipleLines`, `InternetService`, `Contract`, `PaymentMethod`

In [ ]:
# --- Mapeamento binário: Yes/No → 1/0 ---
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}).astype(int)

# --- Colunas com 3 categorias: Yes/No/No internet service ---
# 'No internet service' é equivalente a 'No' para fins preditivos
three_val_cols = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in three_val_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0, 'No internet service': 0}).astype(int)

# Coluna com 3 categorias: Yes/No/No phone service
df['MultipleLines'] = df['MultipleLines'].map({'Yes': 1, 'No': 0, 'No phone service': 0}).astype(int)

# --- One-Hot Encoding para variáveis nominais com múltiplas categorias ---
df = pd.get_dummies(df, columns=['InternetService', 'Contract', 'PaymentMethod'], drop_first=True)

print(f'Shape após encoding: {df.shape}')
print(f'Colunas: {list(df.columns)}')

# 6. Seleção de Features

**Observação do Professor:**
Após o encoding extensivo, nosso dataset expandiu significativamente. Usamos a **`f_classif` (ANOVA F-test)** para selecionar as features mais discriminativas entre churners e não-churners.

A `f_classif` é a escolha ideal aqui pois:
1. Lida com valores negativos (introduzidos pelo `StandardScaler`).
2. É adequada para classificação binária com features contínuas e binárias.
3. Calcula a variância **entre grupos** dividida pela variância **dentro dos grupos** — identificando quais features melhor separam as duas classes de Churn.

Esperamos que features como `tenure`, `Contract_Two year`, `InternetService_Fiber optic` e `TechSupport` tenham alto poder discriminativo, pois clientes mais novos, com internet de fibra cara e sem suporte técnico tendem a cancelar mais.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [ ]:
selector = SelectKBest(score_func=f_classif, k=8)
X_new = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print('Features selecionadas:')
print(selected_features.tolist())

# 7. Split do Dataset

**Observação do Professor - Prevenção ao Vazamento de Dados (Data Leakage):**

Chegamos à separação do dataset! Usamos `stratify=y` para garantir que a proporção de churners (~27%) seja preservada tanto no treino quanto no teste — especialmente importante em datasets desbalanceados.

⚠️ **Lembrete para a prática:** Em produção real, o `StandardScaler` deve ser *fitado* apenas no conjunto de treino e *aplicado* no teste. Em desafios mais avançados do curso, usaremos os `Pipelines` do `scikit-learn` para eliminar esse risco de Data Leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X[selected_features], y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape} | Teste: {X_test.shape}')
print(f'Distribuição do target no treino:\n{y_train.value_counts()}')
print(f'Percentual de Churn no treino: {y_train.mean()*100:.1f}%')

# Balanceamento com SMOTE

**Observação do Professor — SMOTE num dataset levemente desbalanceado:**

O IBM Telco Churn possui um **desbalanceamento leve**: ~73% `No` e ~27% `Yes`. Isso é menos severo que o Churn Bancário (~80/20), mas ainda pode afetar negativamente modelos sensíveis ao desbalanceamento.

Aplicamos o SMOTE como prática do pipeline completo e para demonstrar o efeito em desbalanceamentos moderados. Em competições e projetos de portfólio, aplicar SMOTE em datasets com essa proporção é uma decisão válida que demonstra maturidade técnica.

**Comparação didática das nossas três bases:**
| Dataset | Desbalanceamento | Necessidade de SMOTE |
|---|---|---|
| Spaceship Titanic | ~50/50 | Baixa (apenas didático) |
| IBM Telco Churn | ~73/27 | Moderada (recomendado) |
| Bank Churn Modelling | ~80/20 | **Alta (obrigatório)** |

⚠️ **Regra Fundamental:** O SMOTE deve ser aplicado **APENAS no conjunto de treino** — nunca no conjunto de teste.

In [ ]:
smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f'Antes do SMOTE:  {X_train.shape}')
print(f'Após o SMOTE:    {X_train_balanced.shape}')
print()
print('Distribuição após SMOTE (treino balanceado):')
print(pd.Series(y_train_balanced).value_counts())
print(f'Percentual de Churn após SMOTE: {pd.Series(y_train_balanced).mean()*100:.1f}%')